In [ ]:
import json
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
 
import numpy as np
import polars as pl
import polars.selectors as cs
 
sys.path.insert(0, "..")
from project import get_all, get_info, log, update_run_json

In [ ]:
ALL = get_all()
 
RUN_KEY = "prank"  # which run name from `runs:` to analyse: "prank" or "clustalo"
WORKERS = 32
 
CORE_GAP_MAX = 0.5  # a column is "core" if < 50% of sequences have a gap there
HOMOPOLYMER_MIN = 5  # mononucleotide run length counted as a homopolymer
 
GAP = ord("-")
BASE_CODES = np.frombuffer(b"ACGT", dtype=np.uint8)  # bases counted as states
G_CODE, C_CODE = ord("G"), ord("C")

In [ ]:
ALL = {
    "plants_plt": {
        "solanaceae": get_info("plants_plt", "solanaceae"),
    }
}

In [ ]:
# ----------------------------------------------------------------------------- I/O
def read_aln(path):
    """Aligned FASTA -> (ids, 2D uint8 array of ASCII codes, shape n x L)."""
    ids, chunks = [], []
    with open(path, "rb") as fh:
        for line in fh:
            line = line.strip()
            if line.startswith(b">"):
                ids.append(line[1:].split()[0].decode())
                chunks.append([])
            elif line:
                chunks[-1].append(line)
    seqs = [b"".join(c).upper() for c in chunks]
    L = len(seqs[0]) if seqs else 0
    if any(len(s) != L for s in seqs):
        raise ValueError(f"{path}: sequences have different lengths")
    M = np.frombuffer(b"".join(seqs), dtype=np.uint8).reshape(len(seqs), L)
    return ids, M
 
 
# ------------------------------------------------------------------------- helpers
def _pair_counts(M, notgap):
    """n x n matrices: shared non-gap positions, and identical non-gap positions."""
    G = notgap.astype(np.float32)
    both = G @ G.T
    same = np.zeros_like(both)
    for c in np.unique(M[notgap]):  # every non-gap symbol (ACGT, N, IUPAC...)
        X = (M == c).astype(np.float32)
        same += X @ X.T
    return same, both
 
 
def _pid(same, both):
    """Identity for every pair i<k."""
    iu = np.triu_indices(len(same), 1)
    s, b = same[iu].astype(np.float64), both[iu].astype(np.float64)
    keep = b > 0
    return s[keep] / b[keep]
 
 
def _longest_true_run(x):
    if not x.any():
        return 0
    d = np.diff(np.concatenate(([0], x.astype(np.int8), [0])))
    return int((np.flatnonzero(d == -1) - np.flatnonzero(d == 1)).max())
 
 
def _mode_first_seen(x):
    """Most common value; ties go to the value seen first (same as Counter)."""
    vals, idx, cnt = np.unique(x, return_index=True, return_counts=True)
    best = cnt == cnt.max()
    return vals[best][np.argmin(idx[best])]
 
 
# ------------------------------------------------------------------------- metrics
def aln_metrics(ids, M):
    """Return (summary dict, per-sequence dict of columns) for one alignment."""
    n, L = M.shape
    notgap = M != GAP
    ulen = notgap.sum(axis=1)
 
    # --- per column: gap fraction, base counts, entropy, states, parsimony-informative
    gapf = 1 - notgap.mean(axis=0)
    eq = M[None, :, :] == BASE_CODES[:, None, None]  # (4, n, L)
    counts = eq.sum(axis=1)  # (4, L)
    total = counts.sum(axis=0)
    has = total > 0
    nstates = (counts > 0).sum(axis=0)
    pis = (counts >= 2).sum(axis=0) >= 2
    with np.errstate(divide="ignore", invalid="ignore"):
        p = counts / total
        ent = -np.where(counts > 0, p * np.log2(p), 0.0).sum(axis=0)
    ent[~has] = np.nan
    core = gapf < CORE_GAP_MAX
    var = core & (nstates > 1)
 
    # majority consensus; ties -> base appearing first down the column (as Counter)
    first_row = eq.argmax(axis=1)
    cons_idx = (counts * (n + 1) + (n - first_row)).argmax(axis=0)
    cons = np.where(has, BASE_CODES[cons_idx], GAP).astype(np.uint8)
 
    # --- pairwise identity (all sequences, and unique sequences only)
    same, both = _pair_counts(M, notgap)
    pid = _pid(same, both)
    ungapped = [row[ng].tobytes() for row, ng in zip(M, notgap)]
    uniq = list({s: i for i, s in enumerate(ungapped)}.values())
    sub = np.ix_(uniq, uniq)
    pid_u = _pid(same[sub], both[sub])
 
    # --- longest run of columns that are gap-free and identical in every sequence
    longest = _longest_true_run(notgap.all(axis=0) & (nstates == 1))
 
    # --- indels: internal gap runs (terminal gaps are boundary differences)
    first = notgap.argmax(axis=1)
    last = L - 1 - notgap[:, ::-1].argmax(axis=1)
    padded = np.zeros((n, L + 2), np.int8)
    padded[:, 1:-1] = ~notgap
    d = np.diff(padded, axis=1)
    r, a = np.nonzero(d == 1)  # gap run starts
    _, b = np.nonzero(d == -1)  # gap run ends (exclusive)
    keep = (a > first[r]) & (b <= last[r])
    r, a, b = r[keep], a[keep], b[keep]
    gaps_per_seq = np.bincount(r, minlength=n)
 
    # homopolymer length at every ungapped position, all sequences at once
    rows_ng = np.nonzero(notgap)[0]
    flat = M[notgap]
    new_run = np.ones(len(flat), bool)
    new_run[1:] = (flat[1:] != flat[:-1]) | (rows_ng[1:] != rows_ng[:-1])
    run_id = np.cumsum(new_run) - 1
    run_len = np.bincount(run_id)[run_id] if len(flat) else np.zeros(0, int)
    offset = np.concatenate(([0], np.cumsum(ulen)[:-1]))
    left = np.cumsum(notgap, axis=1)[r, a]  # ungapped bases before the gap
 
    def homo_at(pos):
        out = np.zeros(len(pos), bool)
        ok = (pos >= 0) & (pos < ulen[r])
        out[ok] = run_len[offset[r][ok] + pos[ok]] >= HOMOPOLYMER_MIN
        return out
 
    is_homo = homo_at(left - 1) | homo_at(left)
    keys = a * (L + 1) + b
    n_events = np.unique(keys).size
    n_homo = np.unique(keys[is_homo]).size
 
    # --- boundaries: does each sequence start/end at the most common column?
    off = (first != _mode_first_seen(first)) | (last != _mode_first_seen(last))
 
    # --- GC
    gc = ((M == G_CODE) | (M == C_CODE)).sum(axis=1)
    nz = ulen > 0
 
    summary = {
        "aln_n_seqs": n,
        "aln_len": L,
        "core_len": int(core.sum()),
        "n_haplotypes": len(set(ungapped)),
        "mean_pid": float(pid.mean()) if len(pid) else None,
        "min_pid": float(pid.min()) if len(pid) else None,
        "mean_pid_uniq": float(pid_u.mean()) if len(pid_u) else None,
        "var_sites": int(var.sum()),
        "pis": int((core & pis).sum()),
        "invariant_frac": float(1 - var.sum() / core.sum()) if core.any() else None,
        "mean_entropy": float(np.nanmean(ent[core])) if core.any() else None,
        "indel_events": n_events,
        "indels_per_seq": float(gaps_per_seq.mean()),
        "longest_invariant_block": longest,
        "homopolymer_indel_frac": n_homo / n_events if n_events else None,
        "boundary_offset_frac": float(off.mean()),
        "gc_mean": float(np.mean(gc[nz] / ulen[nz])),
    }
 
    # --- per sequence: identity to the majority consensus
    mask = notgap & has
    msum = mask.sum(axis=1)
    match = ((M == cons) & mask).sum(axis=1)
    per_seq = {
        "seq_id": ids,
        "ungapped_len": ulen.tolist(),
        "id_to_consensus": [float(m / s) if s else None for m, s in zip(match, msum)],
        "internal_gaps": gaps_per_seq.tolist(),
        "boundary_offset": off.tolist(),
    }
    return summary, per_seq
 
 
def process_one(gp, run_dir):
    aln_file = run_dir / f"gp_{gp}_aligned.fasta"
    if not aln_file.exists():
        return gp, None, None
    ids, M = read_aln(aln_file)
    summary, per_seq = aln_metrics(ids, M)
    return gp, summary, per_seq


In [ ]:
for g, subs in ALL.items():
    for subg, info in subs.items():
        p = info["paths"]
        aln_run = info["runs"][RUN_KEY]
        run_dir = p["alignments"] / aln_run
        run_json = run_dir / "run.json"
        log(f"{g}/{subg} | alignment run: {aln_run} ({RUN_KEY})")
 
        if not p["gene_pairs_filtered"].exists():
            log(f"  no {p['gene_pairs_filtered'].name}, run 02_gene_pairs first; skipping")
            continue
        if not run_json.exists():
            log(f"  no {run_json.relative_to(p['dir'])}, skipping")
            continue
        aln_info = json.loads(run_json.read_text())
        if not str(aln_info.get("status", "")).startswith("complete"):
            log(f"  WARNING: run status is {aln_info.get('status')!r}; metrics cover whatever alignments exist")
 
        # gene pairs this run aligned: the filtered list minus the run's own exclusions
        gp_df = pl.read_csv(p["gene_pairs_filtered"], separator="\t")
        excluded = set(aln_info.get("excluded") or [])
        pairs = [gp for gp in gp_df["gene_pair"].to_list() if gp not in excluded]
 
        results, missing = {}, []
        with ThreadPoolExecutor(max_workers=WORKERS) as executor:
            tasks = [executor.submit(process_one, gp, run_dir) for gp in pairs]
            for future in as_completed(tasks):
                gp, summary, per_seq = future.result()
                if summary is None:
                    missing.append(gp)
                else:
                    results[gp] = (summary, per_seq)
 
        # collect in the gene_pairs_filtered.tsv order (threads finish in any order)
        summaries = []
        per_seq_cols = {"gene_pair": []}
        for gp in pairs:
            if gp not in results:
                continue
            summary, per_seq = results[gp]
            summaries.append({"gene_pair": gp, **summary})
            per_seq_cols["gene_pair"] += [gp] * summary["aln_n_seqs"]
            for k, v in per_seq.items():
                per_seq_cols.setdefault(k, []).extend(v)
 
        if not summaries:
            log("  no alignments found, skipping outputs")
            continue
 
        by_pair = (
            gp_df.join(pl.DataFrame(summaries, infer_schema_length=None), on="gene_pair", how="inner")
            .with_columns(cs.float().round(4))
        )
        by_seq = pl.DataFrame(per_seq_cols).with_columns(cs.float().round(4))
        by_pair.write_csv(run_dir / "metrics_by_pair.tsv", separator="\t")
        by_seq.write_csv(run_dir / "metrics_by_seq.tsv", separator="\t")
 
        update_run_json(
            run_json,
            metrics={
                "date": f"{datetime.now():%Y-%m-%d %H:%M:%S}",
                "core_gap_max": CORE_GAP_MAX,
                "homopolymer_min": HOMOPOLYMER_MIN,
                "n_gene_pairs": len(summaries),
                "missing": sorted(missing),
                "files": ["metrics_by_pair.tsv", "metrics_by_seq.tsv"],
            },
        )
 
        log(
            f"  {len(summaries)}/{len(pairs)} gene pairs -> metrics_by_pair.tsv ({by_pair.height} rows),"
            f" metrics_by_seq.tsv ({by_seq.height} rows)" + (f" | excluded: {len(excluded)}" if excluded else "")
        )
        if missing:
            log(f"  missing alignments ({len(missing)}): {missing}")
        print(
            by_pair.select("gene_pair", "aln_n_seqs", "aln_len", "mean_pid", "var_sites", "pis", "indel_events")
            .sort("mean_pid")
            .head(10)
        )